# Single-session performance — the random timeout / banishment task

Log-only performance analysis for ONE session, when all you have is `log.json`. This task puts
**one punishment icon** on the board that is **randomly a banishment or a timeout** (its `options`
field lists the pair `[['banish','fountains'],['timeout','target']]`, and the two have DIFFERENT
textures, so the mouse can see which it is), alongside the reward(s).

What this notebook makes (all from the log):

1. a **clean per-collection dataframe** (like the `results/` tables, but for a single session);
2. the **stochastic p-value over all trials** — is he collecting rewards above the world's chance ratio;
3. the **stochastic p-value for the timeout icon only**;
4. **how many times each effect was collected** (reward / timeout / banishment / escape);
5. the **distance and time between collections**;
6. the **path occupancy around each collected effect** (icon-centred);
7. the **heat map centred on a collected icon type**;
8. the **heading error toward each icon, per trial**, in the trial-report style.

Reuses `common/perf_from_log.py` (stochastic p, world opportunity), `common/geom.py`,
`common/heading.py`, `common/viewport.py`. Nothing needs the video.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# locate common/ (works from results/ in the repo OR from a session folder like flow_test/mouse2/)
_HERE = Path.cwd()
def _find_common():
    for base in (_HERE, *_HERE.parents):
        for rel in ('common', 'session_pipeline/common'):
            if (base / rel / 'perf_from_log.py').exists():
                return base / rel
    return None
_COMMON = _find_common()
assert _COMMON is not None, f'cannot locate common/ from {_HERE}'
sys.path.insert(0, str(_COMMON))
import perf_from_log as pfl
import geom, heading, viewport as vp
print('common ->', _COMMON)

In [ ]:
# ── CONFIG: point at ONE session folder that contains log.json ─────────────────
SESSION_DIR = Path('.')            # <-- the folder with log.json (mouse2 copy: leave as '.')
VIEW_SCALE  = getattr(pfl, 'DEFAULT_VIEW_SCALE', 0.35)   # world zoom (not in the log; 0.35 default)

SESSION_DIR = SESSION_DIR.resolve()
LOGP = SESSION_DIR / 'log.json'
assert LOGP.exists(), f'no log.json in {SESSION_DIR}'
L = json.load(open(LOGP))
W = float(L['worlds'][0].get('width', 2400)); H = float(L['worlds'][0].get('height', 2400))
sess = {'view_scale': VIEW_SCALE, 'world_width': W, 'world_height': H}

# effect valence (effect-level, from perf_from_log)
POS, NEG, NEU = pfl.POSITIVE_EFFECTS, pfl.NEGATIVE_EFFECTS, pfl.NEUTRAL_EFFECTS
def valence(e): return 'positive' if e in POS else 'negative' if e in NEG else 'neutral' if e in NEU else 'other'

t_xy, ax_, ay_ = geom.load_coords(L)           # avatar trajectory (t_ms, x, y)
t_th, theta = geom.load_theta(L)               # avatar heading (t_ms, rad)
theta_t = np.column_stack([t_th, theta]); coords = np.column_stack([t_xy, ax_, ay_])
ed = L.get('experiment_data', {})
print('mouse   :', ed.get('ID'), '| datetime:', ed.get('datetime'))
print('world   :', f'{W:.0f}x{H:.0f}', '| view_scale', VIEW_SCALE, '| collections', len(L['collected']))
import collections as _c
print('effects :', dict(_c.Counter(c['effect'] for c in L['collected'])))

## 1 — the clean per-collection dataframe

One row per collection (`trial = one collected icon`), with the disjoint window `(previous collection,
this collection]`: the collected effect + valence, its position, the reward multiplier, whether the
punishment was randomly assigned (`random_punish` = the `options` field was set), and the per-window
geometry (path efficiency / time in corner / mean speed / heading alignment toward the collected icon),
using the same `common/geom` functions the `results/` tables use.

In [ ]:
coll = L['collected']
rows = []
prev_t = 0.0
for i, c in enumerate(coll):
    tc = float(c['time']); s_ms, e_ms = prev_t, tc
    tt, xx, yy = geom.slice_track(t_xy, ax_, ay_, s_ms, e_ms)
    msp, _ = geom.speed_stats(tt, xx, yy)
    _, align = geom.heading_to_target(t_th, theta, t_xy, ax_, ay_, (c['x'], c['y']), s_ms, e_ms)
    dist_prev = float(np.hypot(c['x'] - coll[i-1]['x'], c['y'] - coll[i-1]['y'])) if i > 0 else np.nan
    rows.append(dict(
        idx=i, effect=c['effect'], valence=valence(c['effect']), texture=c.get('texture'),
        x=float(c['x']), y=float(c['y']), loc=c.get('loc'), time_ms=tc,
        dt_prev_ms=tc - prev_t, dist_prev=dist_prev,
        multiplier=c.get('multiplier'), duration=c.get('duration'),
        random_punish=c.get('options') is not None,
        path_efficiency=geom.path_efficiency(xx, yy),
        time_in_corner=geom.time_in_corner(xx, yy, W, H, t=tt),
        mean_speed=msp, heading_align=align, start_ms=s_ms, end_ms=e_ms))
    prev_t = tc
df = pd.DataFrame(rows)
OUTP = SESSION_DIR / 'clean_trials_log.pkl'
df.to_pickle(OUTP); print('saved', OUTP, '| shape', df.shape)
df.head(12)

## 2 — stochastic p-values

The **world opportunity** baseline: the board offers `active_benefits` good vs `active_detriments` bad
icons, so with NO discrimination the expected hit rate is `benefits/(benefits+detriments)`. The
**stochastic p** = P(≥ the observed number of rewards, if each choice were good with that probability)
— a small p means he collects rewards **above** chance (he discriminates); a large p means at/below
chance. `(1)` pools both punishments; `(2)` counts only the **timeout** icon as the bad option
(reward-vs-timeout), and `(3)` only banishment, so you can see whether he avoids one more than the other.

In [ ]:
ben, det, chance = pfl.world_opportunity(L)
n_pos = int((df.valence == 'positive').sum())
n_timeout = int((df.effect == 'timeout').sum()); n_banish = int((df.effect == 'banish').sum())
n_neg = int((df.valence == 'negative').sum())

def stoch(good, bad, label):
    x = good + bad
    p = pfl.prob_at_least(good, x, chance)
    print(f'  {label:28s} reward {good:2d} vs bad {bad:2d}  (rate {good/x:.3f}, chance {chance:.3f})  '
          f'stochastic p = {p:.4f}')
    return p

print(f'world: {ben} benefit / {det} detriment icons  ->  chance ratio = {chance:.3f}\n')
p_all     = stoch(n_pos, n_neg,     '(1) ALL punishments')
p_timeout = stoch(n_pos, n_timeout, '(2) TIMEOUT only')
p_banish  = stoch(n_pos, n_banish,  '(3) BANISHMENT only')
print('\n(escape/unbanish collections are NEUTRAL and excluded from every rate)')

# cross-check with the full log-only scorecard (note: it classifies this board as banish_multiplier)
try:
    R = pfl.score_log(str(LOGP), view_scale=VIEW_SCALE)
    print('\nscore_log cross-check:',
          {k: (round(R[k], 3) if isinstance(R.get(k), float) else R.get(k))
           for k in ('task', 'observed', 'chance_game_design', 'D_game_design', 'p_game_design') if k in R})
except Exception as e:
    print('\n(score_log skipped:', e, ')')

## 3 — how many times each effect was collected

In [ ]:
order = ['single_reward', 'timeout', 'banish', 'unbanish']
cnt = df['effect'].value_counts().reindex(order).fillna(0).astype(int)
# Okabe-Ito colour-blind-safe palette: reward green, the two hazards blue/vermillion, escape grey
colr = {'single_reward': '#009E73', 'banish': '#0072B2', 'timeout': '#D55E00', 'unbanish': '#999999'}
elabel = {'single_reward': 'reward', 'banish': 'banishment', 'timeout': 'timeout', 'unbanish': 'escape'}
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(range(len(cnt)), cnt.values, color=[colr[e] for e in cnt.index])
ax.set_xticks(range(len(cnt))); ax.set_xticklabels(['reward', 'timeout', 'banishment', 'escape'])
for b, v in zip(bars, cnt.values): ax.text(b.get_x()+b.get_width()/2, v+0.3, str(v), ha='center', fontsize=11)
ax.set_ylabel('times collected'); ax.set_title(f"{ed.get('ID')}  collections by effect  (n={len(df)})")
plt.tight_layout(); plt.show()

## 4 — distance & time between collections

How far apart (icon-to-icon world units) and how long (seconds) between one collection and the next.

In [ ]:
d = df.iloc[1:]                                   # first has no predecessor
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(d.dt_prev_ms/1000, bins=25, color='#8e44ad'); ax[0].set_xlabel('time since previous (s)')
ax[0].set_title(f'inter-collection time  (median {np.median(d.dt_prev_ms)/1000:.1f}s)')
ax[1].hist(d.dist_prev, bins=25, color='#16a085'); ax[1].set_xlabel('distance from previous icon (wu)')
ax[1].set_title(f'inter-collection distance  (median {np.nanmedian(d.dist_prev):.0f} wu)')
for e in ['single_reward', 'timeout', 'banish', 'unbanish']:
    s = d[d.effect == e]
    if len(s): ax[2].scatter(s.dt_prev_ms/1000, s.dist_prev, s=28, color=colr[e], label={'single_reward':'reward','banish':'banishment','unbanish':'escape'}.get(e, e))
ax[2].set_xlabel('time (s)'); ax[2].set_ylabel('distance (wu)'); ax[2].legend(fontsize=8); ax[2].set_title('distance vs time, by effect')
plt.tight_layout(); plt.show()

## 4b — distance & time between collections, per collected type

The same inter-collection time and distance as above, but split by the **type** he just collected —
so you can compare how he arrives at a reward vs a banishment vs a timeout. If banishment looks like
reward here (and in the icon-centred occupancy in section 5), he is approaching the hazard the same way
he approaches a reward, i.e. not avoiding it.

In [ ]:
types = [(e, elabel[e], colr[e]) for e in ('single_reward', 'banish', 'timeout')]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for j, (e, lbl, col) in enumerate(types):
    s = df[(df.effect == e) & (df.idx > 0)]
    ax[0, j].hist(s.dt_prev_ms/1000, bins=15, color=col)
    ax[0, j].set_title(f'{lbl}: time since previous\n(median {np.nanmedian(s.dt_prev_ms)/1000:.1f}s, n={len(s)})')
    ax[0, j].set_xlabel('time (s)')
    ax[1, j].hist(s.dist_prev, bins=15, color=col)
    ax[1, j].set_title(f'{lbl}: distance from previous\n(median {np.nanmedian(s.dist_prev):.0f} wu)')
    ax[1, j].set_xlabel('distance (wu)')
plt.suptitle('inter-collection TIME (top) and DISTANCE (bottom), by collected type'); plt.tight_layout(); plt.show()


# box-plot comparison across ALL collected types (side by side)
cats = [e for e in ('single_reward', 'banish', 'timeout', 'unbanish') if (df[df.idx > 0].effect == e).any()]
figb, axb = plt.subplots(1, 2, figsize=(12, 4.2))
rng = np.random.default_rng(0)
for a, (col, ylab, sc) in zip(axb, [('dt_prev_ms', 'time since previous (s)', 1e-3),
                                    ('dist_prev', 'distance from previous (wu)', 1.0)]):
    data = [df[(df.effect == e) & (df.idx > 0)][col].dropna().values * sc for e in cats]
    bp = a.boxplot(data, patch_artist=True, showfliers=False, widths=0.6)
    for patch, e in zip(bp['boxes'], cats): patch.set_facecolor(colr[e]); patch.set_alpha(.75)
    for med in bp['medians']: med.set_color('k')
    for i, v in enumerate(data):                       # overlay the individual collections
        a.scatter(np.full(len(v), i + 1) + rng.uniform(-.12, .12, len(v)), v, s=14, color='0.2', alpha=.5, zorder=3)
    a.set_xticks(range(1, len(cats) + 1)); a.set_xticklabels([elabel[e] for e in cats])
    a.set_ylabel(ylab); a.set_title(ylab.split(' (')[0] + ' by type')
plt.suptitle('inter-collection time & distance — box-plot comparison across icons'); plt.tight_layout(); plt.show()

# direct comparison: is the approach to a banishment the same as to a reward?
print('per-type approach (from the previous collection):')
print(df[df.idx > 0].groupby('effect').agg(
    n=('effect', 'size'), time_s=('dt_prev_ms', lambda v: round(np.nanmedian(v)/1000, 1)),
    dist_wu=('dist_prev', lambda v: round(np.nanmedian(v))),
    path_eff=('path_efficiency', lambda v: round(np.nanmedian(v), 3)),
    heading_align=('heading_align', lambda v: round(np.nanmean(v), 3))).to_string())

## 4c — the multiplier: how high did the reward streaks build?

Consecutive rewards build the **multiplier** (1→4, resetting on any negative). This is descriptive:
(a) the multiplier over the whole session, each collection coloured by what it was, and (b) how often
he reached each multiplier level. The *test* of whether he actually prefers/exploits it is in 4d.

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
mv = df['multiplier'].astype(float).fillna(0).values
ax[0].plot(range(len(df)), mv, '-', color='0.85', lw=.8, zorder=1)
ax[0].scatter(range(len(df)), mv, c=[colr.get(e, '0.6') for e in df['effect']], s=24, zorder=2)
handles = [Line2D([0], [0], marker='o', ls='', color=colr[e], label=elabel[e])
           for e in ['single_reward', 'banish', 'timeout', 'unbanish'] if (df.effect == e).any()]
ax[0].legend(handles=handles, fontsize=8, loc='upper left', ncol=2)
ax[0].set_title('(a) multiplier over the session'); ax[0].set_xlabel('collection #'); ax[0].set_ylabel('multiplier')
mult = df.loc[df.valence == 'positive', 'multiplier'].dropna().astype(int)
vc = mult.value_counts().sort_index()
ax[1].bar(vc.index, vc.values, color=colr['single_reward'])
for x, v in zip(vc.index, vc.values): ax[1].text(x, v + 0.3, str(v), ha='center', fontsize=9)
ax[1].set_title(f'(b) multiplier reached at reward collections'); ax[1].set_xlabel('multiplier'); ax[1].set_ylabel('# reward collections')
plt.suptitle('multiplier: how high did the reward streaks build?'); plt.tight_layout(); plt.show()
print(f'mean multiplier at rewards: {mult.mean():.2f}, max {int(mult.max())}')

## 4d — decision-making test: does he prefer / exploit the multiplier?

The standard 2-choice sequential test. Treat every collection as a choice between **good** (reward) and
**bad** (a negative — banishment *or* timeout pooled); escapes are neutral and excluded. Three readings,
all in plain terms:

- **(a) Win-stay** — after collecting a reward, how often is his *next* collection also a reward, versus
  after hitting a negative? If "after a reward" is clearly higher, he perseveres on reward (win-stay),
  the signature of exploiting a streak reward. (Fisher exact test on the 2×2.)
- **(b) Escalation** — does `P(next = reward)` climb the longer his current reward run is? (Wilson 95%
  intervals; read the intervals, not the dots — n shrinks fast at long runs.)
- **(c) Earnings, in reward-drops** — he *earned* a number of drops (each reward pays its multiplier).
  Compared with two fixed references: **no streaks at all** (every reward = 1 drop) and **one big streak**
  (the most the multiplier could ever give these rewards). *Bonus captured = (earned − no-streak) /
  (max − no-streak)*: **0 % = the multiplier never helped him, 100 % = he used it maximally.** This is a
  concrete number, not a shuffle.

In [ ]:
import streaks
from scipy.stats import binomtest, fisher_exact
seq = df[df.valence.isin(['positive', 'negative'])].valence.eq('positive').values   # True = reward
st = streaks.analyse(seq, mult_cap=4)
base = float(seq.mean())
cur, nxt = seq[:-1], seq[1:]
p_after_good = float(nxt[cur].mean());  n_good = int(cur.sum())
p_after_bad  = float(nxt[~cur].mean()); n_bad  = int((~cur).sum())
a, b = int((cur & nxt).sum()), int((cur & ~nxt).sum())
c_, d_ = int((~cur & nxt).sum()), int((~cur & ~nxt).sum())
_, p_fish = fisher_exact([[a, b], [c_, d_]])

def drops(s, cap=4):
    tot, m = 0, 1
    for x in s:
        if x: tot += m; m = min(m + 1, cap)
        else: m = 1
    return tot
n_pos = int(seq.sum())
flat, best, earned = n_pos, drops(np.ones(n_pos, bool)), st['drops_earned']
captured = (earned - flat) / (best - flat) if best > flat else np.nan

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
ax[0].bar([0, 1], [p_after_good, p_after_bad], color=[colr['single_reward'], colr['banish']])
ax[0].axhline(base, ls='--', color='k', label=f'overall reward rate {base:.2f}')
for x, v in zip([0, 1], [p_after_good, p_after_bad]): ax[0].text(x, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels([f'after a REWARD\n(n={n_good})', f'after a NEGATIVE\n(n={n_bad})'])
ax[0].set_ylim(0, 1.08); ax[0].set_ylabel('P(next collection = reward)')
ax[0].set_title(f'(a) win-stay  (Fisher p={p_fish:.3f})'); ax[0].legend(fontsize=8)
ks = sorted(st['cond']); pk = [st['cond'][k]['p'] for k in ks]
lo = [st['cond'][k]['p'] - st['cond'][k]['lo'] for k in ks]; hi = [st['cond'][k]['hi'] - st['cond'][k]['p'] for k in ks]
ax[1].errorbar(ks, pk, yerr=[lo, hi], marker='o', capsize=3, color=colr['single_reward'])
for k in ks: ax[1].annotate(f"n={st['cond'][k]['n']}", (k, st['cond'][k]['p']), fontsize=7, textcoords='offset points', xytext=(0, 8), ha='center')
ax[1].axhline(base, ls='--', color='k'); ax[1].set_ylim(0, 1.1)
ax[1].set_xlabel('rewards collected in a row so far'); ax[1].set_ylabel('P(next = reward)'); ax[1].set_title('(b) escalation with the streak')
ax[2].bar([0, 1, 2], [flat, earned, best], color=['0.75', colr['single_reward'], '0.4'])
ax[2].set_xticks([0, 1, 2]); ax[2].set_xticklabels([f'no streaks\n{flat}', f'HIS order\n{earned}', f'max streak\n{best}'])
ax[2].set_ylabel('reward drops earned'); ax[2].set_title(f'(c) multiplier bonus captured = {captured*100:.0f}%')
plt.suptitle(f"{ed.get('ID')}  decision-making: does he prefer / exploit the multiplier?"); plt.tight_layout(); plt.show()

print(f'WIN-STAY : P(reward | just collected a reward) = {p_after_good:.3f}  (n={n_good})')
print(f'           P(reward | just hit a negative)     = {p_after_bad:.3f}  (n={n_bad})   Fisher p={p_fish:.3f}')
print(f'EARNINGS : earned {earned} drops; no-streak floor {flat}, max-streak ceiling {best}'
      f'  ->  captured {captured*100:.0f}% of the multiplier bonus')
print(f'RUNS TEST: {streaks.verdict(st)}')

## 5 — path occupancy around each collected effect (icon-centred)

**How it is computed.** For every collection of a given effect, take the avatar `(x, y)` samples logged
within **±`window_s` (default 3 s)** of that collection's timestamp, and subtract the icon's world
position, so the icon sits at the origin `(0, 0)`. Pool these **icon-centred offsets** across *all*
collections of that type and bin them into a 2-D histogram: each cell counts how many avatar samples
landed there — i.e. how much time he spent at that offset from the icon (`coords` are logged while he
moves, so a bright cell = he passed through or lingered there often). The star at the origin is the
icon; the bright ring at ~200 world units is the collection radius (he has to reach the icon to collect
it). Comparing the maps tells you whether he approaches a hazard from all sides (like a reward he does
not avoid) or from a restricted direction.

In [ ]:
def offsets_around(effect, window_s=3.0):
    ox, oy = [], []
    for _, r in df[df.effect == effect].iterrows():
        m = (t_xy >= r.time_ms - window_s*1000) & (t_xy <= r.time_ms + window_s*1000)
        ox.append(ax_[m] - r.x); oy.append(ay_[m] - r.y)
    return (np.concatenate(ox), np.concatenate(oy)) if ox else (np.array([]), np.array([]))

def occupancy(effect, half=900, bins=45, window_s=3.0, ax=None, title=None):
    ox, oy = offsets_around(effect, window_s)
    ax = ax or plt.gca()
    if ox.size:
        Hh, xe, ye = np.histogram2d(ox, oy, bins=bins, range=[[-half, half], [-half, half]])
        ax.imshow(Hh.T, origin='lower', extent=[-half, half, -half, half], cmap='magma', aspect='equal')
    ax.plot(0, 0, marker='*', ms=16, color='cyan', mec='k')       # the collected icon at centre
    ax.set_title(title or effect); ax.set_xlabel('x - icon (wu)'); ax.set_ylabel('y - icon (wu)')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for a, (e, lbl) in zip(axes, [('single_reward', 'reward'), ('banish', 'banishment'), ('timeout', 'timeout')]):
    occupancy(e, ax=a, title=f'{lbl}  (n={int((df.effect==e).sum())})')
plt.suptitle('path occupancy around the collected icon (±3 s, icon at centre)'); plt.tight_layout(); plt.show()

## 6 — heat map centred on one collected icon type

The same computation, zoomed to one type with a colour bar — pick `ICON` and the window. The star is
the icon; brighter = the avatar spent more time there (occupancy).

In [ ]:
ICON = 'single_reward'      # <-- 'single_reward' | 'banish' | 'timeout' | 'unbanish'
WINDOW_S, HALF = 3.0, 900
ox, oy = offsets_around(ICON, WINDOW_S)
fig, ax = plt.subplots(figsize=(6.4, 5.6))
if ox.size:
    Hh, xe, ye = np.histogram2d(ox, oy, bins=45, range=[[-HALF, HALF], [-HALF, HALF]])
    im = ax.imshow(Hh.T, origin='lower', extent=[-HALF, HALF, -HALF, HALF], cmap='magma', aspect='equal')
    fig.colorbar(im, ax=ax, label='avatar samples (occupancy)')
ax.plot(0, 0, marker='*', ms=18, color='cyan', mec='k', label=f'{ICON} (collected)')
ax.set_title(f"{ed.get('ID')}  occupancy centred on {ICON}  (±{WINDOW_S}s, n={int((df.effect==ICON).sum())})")
ax.set_xlabel('x - icon (world units)'); ax.set_ylabel('y - icon (world units)'); ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 7 — heading error toward each icon, per trial (trial-report style)

For each trial (collection window) the heading error is `cos(angle between where he points and the
bearing to the icon)`: **+1 = straight at it, 0 = sideways, −1 = away**, plotted against time to the
collection (`t = 0`). The **collected icon** is the goal (solid); the other icons on the board that
trial (from the spawn snapshot) are shown too, so you can see whether he was oriented to the punishment.

In [ ]:
spawns = sorted(L.get('spawns', []), key=lambda s: s['time'])
def board_at(tc):
    '''icons on the board at time tc = the latest spawn batch's `current` snapshot.'''
    cur = []
    for sp in spawns:
        if sp['time'] <= tc: cur = sp.get('current') or cur
        else: break
    return [ic for ic in cur if ic.get('effect') in (POS | NEG)]

def heading_page(idx, ax=None):
    r = df.iloc[int(idx)]; s_ms, e_ms = r.start_ms, r.end_ms
    ax = ax or plt.gca()
    tt, err = heading.heading_error(theta_t, coords, (r.x, r.y), s_ms, e_ms)
    if tt.size:
        ax.plot((tt - e_ms)/1000, np.cos(err), lw=2, color=colr.get(r.effect, 'k'),
                label=f'collected {r.effect}', zorder=3)
    for ic in board_at(e_ms):
        if ic['x'] == r.x and ic['y'] == r.y: continue
        tt2, err2 = heading.heading_error(theta_t, coords, (ic['x'], ic['y']), s_ms, e_ms)
        if tt2.size:
            ax.plot((tt2 - e_ms)/1000, np.cos(err2), lw=1, alpha=.6, ls='--',
                    color=colr.get(ic['effect'], '0.5'), label=f"{ic['effect']}")
    ax.axhline(0, color='0.7', lw=.6); ax.axvline(0, color='0.7', lw=.6)
    ax.set_ylim(-1.1, 1.1); ax.set_title(f"trial {int(idx)}: {r.effect}", fontsize=9)
    ax.set_xlabel('time to collection (s)'); ax.set_ylabel('cos(heading error)')
    h, l = ax.get_legend_handles_labels(); ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), fontsize=6)

# a few example trials: one of each effect (last window >= 2 s)
ex = []
for e in ('single_reward', 'banish', 'timeout'):
    s = df[(df.effect == e) & (df.dt_prev_ms > 2000)]
    if len(s): ex.append(int(s.iloc[len(s)//2].idx))
fig, axes = plt.subplots(1, len(ex), figsize=(5.2*len(ex), 4))
for a, i in zip(np.atleast_1d(axes), ex): heading_page(i, a)
plt.suptitle('heading error toward each icon (trial-report style)'); plt.tight_layout(); plt.show()

# summary: mean forward-alignment toward the COLLECTED icon, by effect
print('mean heading alignment toward the collected icon, by effect:')
print(df.groupby('effect')['heading_align'].agg(['mean', 'count']).round(3).to_string())